# 06 — Coupling-graph routing

Yen *k*-shortest paths from 5lpg ligand contacts to low-scoring deconvolution positions on the shipped EVcouplings graph.


In [ ]:
from pathlib import Path

import pandas as pd

from dualde.contacts import NUDT15_LIGAND_NAMES, attach_minimum_distance
from dualde.paths import example_dir
from dualde.routing import run_information_transfer_pipeline

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

deconv = pd.read_csv(OUT / "deconvolution.csv")
contacts = pd.read_csv(OUT / "5lpg_contacts.csv")
func_path = OUT / "5lpg_functional_residues.csv"
couplings = EXAMPLE / "couplings" / "b0.5_CouplingScoresCompared_all.csv"

scores = attach_minimum_distance(deconv, contacts)

score_col = "EVmutation_neg_spurs_model"
if score_col not in scores.columns:
    candidates = [c for c in scores.columns if c.endswith("_neg_spurs_model")]
    if not candidates:
        raise ValueError("No deconvolution model column found; run notebook 05 first.")
    score_col = candidates[0]
print("score_col:", score_col)

result = run_information_transfer_pipeline(
    all_merged_df=scores,
    couplings_file=str(couplings),
    functional_residues_file=str(func_path),
    active_site_ligands=list(NUDT15_LIGAND_NAMES),
    activity_wt=0.45,
    abundance_wt=0.45,
    score_col=score_col,
    low_score_percentile=1.0,
    prob_threshold=0.7,
    n_paths=5,
    weighted=True,
    dist_cutoff=5,
    seq_len=164,
    verbose=True,
)

result["paths_df"].to_csv(OUT / "coupling_paths.csv", index=False)
result["pairs_df"].to_csv(OUT / "coupling_pairs.csv", index=False)
print("n paths", len(result["paths_df"]))
result["paths_df"].head()
